# Azure Private Link & Snowflake — End-to-End Setup

This notebook walks through every step required to configure **Azure Private Link** for private connectivity between your Azure Virtual Network (VNet) and your Snowflake account on Azure.

Azure Private Link ensures that all traffic between your VNet and Snowflake traverses the **Microsoft backbone network**, never crossing the public internet.

```
┌───────────────────────────────────────────────────────────────────────┐
│                    YOUR AZURE ENVIRONMENT                        │
│                                                                       │
│  ┌───────────────────────────────────────────────────────────────┐  │
│  │                   Virtual Network (VNet)                     │  │
│  │                                                               │  │
│  │  ┌───────────────┐   ┌─────────────────────────────────────┐  │  │
│  │  │  VM / App    │   │  Private Endpoint (PE)              │  │  │
│  │  │  Service     │──▶│  (NIC with Private IP: 10.x.x.x)   │  │  │
│  │  └───────────────┘   └──────────────────┴──────────────────┘  │  │
│  │                                        │                        │  │
│  │  ┌───────────────────────────┐       │                        │  │
│  │  │ Private DNS Zone          │       │  Microsoft Backbone   │  │
│  │  │ privatelink.              │       │  (no public internet) │  │
│  │  │   snowflakecomputing.com  │       │                        │  │
│  │  └───────────────────────────┘       │                        │  │
│  └───────────────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────┴─────────────────────────┘
                                              │
                                     ┌─────────┴──────────────┐
                                     │  SNOWFLAKE VNet          │
                                     │  ┌────────────────────┐  │
                                     │  │ Private Link       │  │
                                     │  │ Service (PLS)      │  │
                                     │  └────────────────────┘  │
                                     │  ┌────────────────────┐  │
                                     │  │ Snowflake Service  │  │
                                     │  └────────────────────┘  │
                                     └─────────────────────────┘
```

### Prerequisites

| Requirement | Details |
|---|---|
| **Snowflake Edition** | Business Critical (or higher) |
| **Snowflake Role** | `ACCOUNTADMIN` |
| **Cloud Platform** | Snowflake account hosted on **Azure** |
| **Azure Subscription** | Active subscription with Owner or Contributor access |
| **Azure CLI** | Installed and authenticated (`az login`) |
| **Azure Permissions** | Ability to create VNets, Private Endpoints, Private DNS Zones |
| **Network** | TCP ports 443 and 80 open in NSG for the Private Endpoint subnet |
| **VNet Type** | ARM VNets only (Classic VNets not supported) |
| **IP Protocol** | IPv4 TCP traffic only |

---
## Part 1 — Snowflake: Pre-flight Checks

Before configuring anything in Azure, we need to gather information from the Snowflake account. This includes confirming the account is on Azure, retrieving the Private Link Service (PLS) ID, and collecting all the URLs that will need DNS records.

### Step 1 — Confirm Account & Region

Verify that this Snowflake account is hosted on Azure and note the region. Azure Private Link only works with Snowflake accounts deployed on Azure.

**Why:** The Private Link Service ID and all URLs are region-specific. You need to know the Azure region to create the Private Endpoint in the same (or peered) region.

> **Impact:** If the account is not on Azure, you cannot use Azure Private Link. Use AWS PrivateLink or GCP Private Service Connect instead.

> **Doc:** [Azure Private Link and Snowflake](https://docs.snowflake.com/en/user-guide/privatelink-azure)

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;

SELECT CURRENT_ACCOUNT()    AS account,
       CURRENT_REGION()     AS region,
       CURRENT_ORGANIZATION_NAME() AS org_name,
       SYSTEM$GET_SNOWFLAKE_PLATFORM_INFO() AS platform_info;

### Step 2 — Retrieve Private Link Configuration

This query retrieves all the Private Link configuration values for this account. The most important value is **`privatelink-pls-id`** — this is the alias for the Snowflake Private Link Service that you will use to create the Azure Private Endpoint.

**Why:** The PLS ID is required to create the Private Endpoint in Azure. The URL values are needed later for DNS configuration.

> **Impact:** Without the PLS ID, you cannot create the Private Endpoint. Without the URLs, DNS resolution will fail and clients won’t reach Snowflake privately.

**Save the output** — you will need these values throughout this notebook.

> **Doc:** [SYSTEM$GET_PRIVATELINK_CONFIG](https://docs.snowflake.com/en/sql-reference/functions/system_get_privatelink_config)

In [ ]:
%%sql -r dataframe_2
SELECT key, value::VARCHAR AS host
FROM TABLE(
    FLATTEN(
        INPUT => PARSE_JSON(SYSTEM$GET_PRIVATELINK_CONFIG())
    )
);

---
## Part 2 — Azure: Create VNet & Private Endpoint

This section covers the Azure-side configuration. You will create a Virtual Network (if you don’t already have one), then create a Private Endpoint that connects to Snowflake’s Private Link Service.

All commands below use the **Azure CLI** (`az`). You can also perform these steps in the **Azure Portal**.

### 2.1 — List Azure Subscriptions

First, identify your Azure subscription ID. You’ll need this for generating the access token later.

```bash
az account list --output table
```

Example output:

```
Name       CloudName    SubscriptionId                        State    IsDefault
---------  -----------  ------------------------------------  -------  ----------
MyCloud    AzureCloud   13c12345-abcd-6789-ef01-234567890abc  Enabled  True
```

Note your **SubscriptionId** — you will need it in step 2.6.

### 2.2 — Create a Resource Group (Optional)

If you don’t already have a resource group for your networking resources, create one:

```bash
az group create \
  --name <RESOURCE_GROUP_NAME> \
  --location <AZURE_REGION>
```

Replace:
- `<RESOURCE_GROUP_NAME>` — e.g., `rg-snowflake-privatelink`
- `<AZURE_REGION>` — must match or peer with the region where Snowflake is deployed (e.g., `eastus2`, `westeurope`)

> **Impact:** The Private Endpoint must be in a region that can reach the Snowflake PLS. Same region is simplest.

### 2.3 — Create a Virtual Network

Create a VNet with a subnet that will host the Private Endpoint.

```bash
az network vnet create \
  --name <VNET_NAME> \
  --resource-group <RESOURCE_GROUP_NAME> \
  --location <AZURE_REGION> \
  --address-prefix 10.0.0.0/16 \
  --subnet-name <SUBNET_NAME> \
  --subnet-prefix 10.0.0.0/24
```

Replace:
- `<VNET_NAME>` — e.g., `vnet-snowflake-privatelink`
- `<SUBNET_NAME>` — e.g., `snet-privatelink`

**Azure Portal path:** Virtual networks → + Create → fill in Basics, IP addresses, Review + Create

**Why:** The Private Endpoint requires a VNet and subnet to attach to. If you already have a VNet, skip this step.

> **Impact:** Ensure TCP ports **443** and **80** are allowed to `0.0.0.0` in the NSG attached to the subnet.

### 2.4 — Create the Private Endpoint

Create a Private Endpoint that connects to the Snowflake Private Link Service using the **`privatelink-pls-id`** value from Step 2.

**Azure Portal path:** Private Link Center → Private endpoints → + Create

| Portal Field | Value |
|---|---|
| **Subscription** | Your Azure subscription |
| **Resource Group** | `<RESOURCE_GROUP_NAME>` |
| **Name** | e.g., `pep-snowflake-privatelink` |
| **Region** | Same region as your VNet |
| **Connection method** | Connect to an Azure resource by **resource ID or alias** |
| **Resource ID or alias** | Paste the `privatelink-pls-id` value from Step 2 |
| **Virtual Network** | Select your VNet from step 2.3 |
| **Subnet** | Select your subnet |

**Azure CLI equivalent:**

```bash
az network private-endpoint create \
  --name pep-snowflake-privatelink \
  --resource-group <RESOURCE_GROUP_NAME> \
  --vnet-name <VNET_NAME> \
  --subnet <SUBNET_NAME> \
  --connection-name snowflake-pl-connection \
  --manual-request true \
  --request-message "Snowflake Private Link" \
  --private-connection-resource-id "<PRIVATELINK_PLS_ID>"
```

Replace `<PRIVATELINK_PLS_ID>` with the `privatelink-pls-id` value (the alias) from Step 2.

> **Impact:** After creation, the endpoint’s **CONNECTION STATE** will show **Pending** until you authorize it in Snowflake (Step 3). A green checkmark on the alias confirms it is valid.

> **Doc:** [Azure Private Endpoint](https://docs.microsoft.com/en-us/azure/private-link/private-endpoint-overview)

### 2.5 — Get the Private Endpoint Resource ID

You need the **resource ID** of the Private Endpoint you just created. This is passed to Snowflake for authorization.

**Azure Portal:** Private endpoints → select your endpoint → Properties → copy **Resource ID**

**Azure CLI:**

```bash
az network private-endpoint show \
  --name pep-snowflake-privatelink \
  --resource-group <RESOURCE_GROUP_NAME> \
  --query id \
  --output tsv
```

The output will look like:

```
/subscriptions/26d.../resourcegroups/rg-snowflake-privatelink/providers/microsoft.network/privateendpoints/pep-snowflake-privatelink
```

**Save this value** — you need it in Step 3.

### 2.6 — Generate an Azure Access Token

Snowflake needs a temporary access token to verify your Azure subscription owns the Private Endpoint.

```bash
az account get-access-token --subscription <SUBSCRIPTION_ID>
```

Example output:

```json
{
  "accessToken": "eyJ...",
  "expiresOn": "2026-04-07 21:38:31.401332",
  "subscription": "0cc...",
  "tenant": "d47...",
  "tokenType": "Bearer"
}
```

**Copy the `accessToken` value** — you need it in Step 3.

> **⚠️ Security:** The access token grants temporary Read access to your Azure subscription. Treat it like a password. It expires within 60 minutes. Never share it in logs or tickets.

> **Permissions:** The user generating the token must have **Reader** permission on the subscription (minimum: `Microsoft.Subscription/subscriptions/acceptOwnershipStatus/read`).

---
## Part 3 — Snowflake: Authorize Private Link

Now that the Azure Private Endpoint exists, we authorize it in Snowflake. This links your Private Endpoint to the Snowflake Private Link Service and changes the endpoint’s connection state from **Pending** to **Approved**.

### Step 3 — Authorize Private Link

Call `SYSTEM$AUTHORIZE_PRIVATELINK` with the Private Endpoint resource ID and the Azure access token from the previous steps.

**Why:** This tells Snowflake to accept connections from your specific Azure Private Endpoint. Without authorization, the endpoint stays in “Pending” state and cannot route traffic.

> **Impact:** If the access token has expired (>60 min), regenerate it with `az account get-access-token` and retry.

Replace the placeholder values below:
- `<PRIVATE_ENDPOINT_RESOURCE_ID>` — from step 2.5
- `<ACCESS_TOKEN>` — from step 2.6

> **Doc:** [SYSTEM$AUTHORIZE_PRIVATELINK](https://docs.snowflake.com/en/sql-reference/functions/system_authorize_privatelink)

In [ ]:
%%sql -r dataframe_3
USE ROLE ACCOUNTADMIN;

SELECT SYSTEM$AUTHORIZE_PRIVATELINK(
    '<PRIVATE_ENDPOINT_RESOURCE_ID>',
    '<ACCESS_TOKEN>'
);

### Step 4 — Verify Authorization

Confirm that Private Link has been successfully authorized for this account.

**Why:** You should see `Account is authorized for PrivateLink.` — this confirms Snowflake will accept connections from your Private Endpoint.

> **Impact:** If authorization fails, verify the Private Endpoint resource ID format and that the access token hasn’t expired.

> **Doc:** [SYSTEM$GET_PRIVATELINK](https://docs.snowflake.com/en/sql-reference/functions/system_get_privatelink)

In [ ]:
%%sql -r dataframe_4
SELECT SYSTEM$GET_PRIVATELINK() AS privatelink_status;

---
## Part 4 — Azure: DNS Configuration

The Private Endpoint is now approved, but clients still need DNS to resolve Snowflake URLs to the Private Endpoint’s private IP address. This is the most critical and commonly misconfigured part of the setup.

There are two DNS approaches:
1. **Azure Private DNS Zone** — for resources inside the VNet (VMs, Functions, etc.)
2. **Local hosts file** — for end-user machines connecting via VPN

### 4.1 — Get the Private Endpoint IP Address

All DNS records will point to this IP address.

**Azure Portal:** Private endpoints → select your endpoint → select the **Network Interface** → note the **Private IP address** (e.g., `10.0.0.5`)

**Azure CLI:**

```bash
# Get the NIC ID of the private endpoint
NIC_ID=$(az network private-endpoint show \
  --name pep-snowflake-privatelink \
  --resource-group <RESOURCE_GROUP_NAME> \
  --query "networkInterfaces[0].id" \
  --output tsv)

# Get the private IP from the NIC
az network nic show \
  --ids $NIC_ID \
  --query "ipConfigurations[0].privateIPAddress" \
  --output tsv
```

**Save this IP address** — you’ll use it for every DNS record below.

### 4.2 — Create a Private DNS Zone

Create a Private DNS Zone for `privatelink.snowflakecomputing.com`. This zone will resolve all Snowflake Private Link URLs to the Private Endpoint’s IP.

**Azure Portal:** Private DNS zones → + Create → Name: `privatelink.snowflakecomputing.com`

**Azure CLI:**

```bash
az network private-dns zone create \
  --resource-group <RESOURCE_GROUP_NAME> \
  --name privatelink.snowflakecomputing.com
```

> **Impact:** Only **one** Private DNS Zone per domain per resource group is allowed. If you have multiple Snowflake accounts, they must be in separate resource groups or share DNS records in one zone.

> **Doc:** [Azure Private DNS Zones](https://docs.microsoft.com/en-us/azure/dns/private-dns-privatednszone)

### 4.3 — Create DNS A Records

Create an **A record** for each URL returned by `SYSTEM$GET_PRIVATELINK_CONFIG()` in Step 2. The record name is everything **before** `.privatelink.snowflakecomputing.com`, and the IP is the Private Endpoint IP from step 4.1.

**URLs that need A records** (from Step 2 output):

| Config Key | Example DNS Record Name | Resolves To |
|---|---|---|
| `privatelink-account-url` | `acctid.azure-region` | `<PE_IP>` |
| `privatelink_ocsp-url` | `ocsp.acctid.azure-region` | `<PE_IP>` |
| `regionless-privatelink-account-url` | `org-account_name` | `<PE_IP>` |
| `regionless-privatelink-ocsp-url` | `ocsp.org-account_name` | `<PE_IP>` |
| `regionless-snowsight-privatelink-url` | `app-org-account_name` | `<PE_IP>` |
| `snowsight-privatelink-url` | `app.azure-region` | `<PE_IP>` |

**Azure CLI** (repeat for each record):

```bash
az network private-dns record-set a add-record \
  --resource-group <RESOURCE_GROUP_NAME> \
  --zone-name privatelink.snowflakecomputing.com \
  --record-set-name <DNS_RECORD_NAME> \
  --ipv4-address <PE_IP>
```

> **Tip:** Also create records with **hyphens** replacing underscores in account names (e.g., `org-my-account` alongside `org-my_account`). Some drivers and tools don’t support underscores in hostnames.

> **Impact:** If any URL is missing a DNS record, that specific Snowflake access path (account URL, Snowsight, OCSP validation) will fail over Private Link.

### 4.4 — Link the DNS Zone to Your VNet

Connect the Private DNS Zone to your VNet so resources inside the VNet can resolve the Snowflake Private Link URLs.

**Azure Portal:** Private DNS zones → select zone → Settings → Virtual network links → + Add

**Azure CLI:**

```bash
az network private-dns link vnet create \
  --resource-group <RESOURCE_GROUP_NAME> \
  --zone-name privatelink.snowflakecomputing.com \
  --name snowflake-vnet-link \
  --virtual-network <VNET_NAME> \
  --registration-enabled false
```

**Why:** Without this link, the DNS zone exists but VNet resources cannot query it. You can link multiple VNets (including peered VNets) to the same DNS zone.

> **Impact:** Resources in VNets without a link will still resolve Snowflake URLs via public DNS, bypassing the Private Endpoint.

### 4.5 — Local DNS for End Users (Optional)

For end users connecting via VPN (not from inside the VNet), update the local **hosts file** to route Snowflake URLs to the Private Endpoint IP.

**Windows:** Edit `C:\Windows\System32\drivers\etc\hosts` (as Administrator)

**macOS / Linux:** Edit `/etc/hosts` (with `sudo`)

Add one line per URL (use actual values from Step 2):

```
# Snowflake Azure Private Link
10.0.0.5 acctid.azure-region.privatelink.snowflakecomputing.com
10.0.0.5 ocsp.acctid.azure-region.privatelink.snowflakecomputing.com
10.0.0.5 org-account_name.privatelink.snowflakecomputing.com
10.0.0.5 org-account-name.privatelink.snowflakecomputing.com
10.0.0.5 ocsp.org-account_name.privatelink.snowflakecomputing.com
10.0.0.5 ocsp.org-account-name.privatelink.snowflakecomputing.com
10.0.0.5 app-org-account_name.privatelink.snowflakecomputing.com
10.0.0.5 app-org-account-name.privatelink.snowflakecomputing.com
10.0.0.5 app.azure-region.privatelink.snowflakecomputing.com
```

> **Impact:** The user must be connected to a VPN that can route to the Private Endpoint’s private IP address. Without VPN access, the hosts file alone won’t work.

---
## Part 5 — Snowflake: Verification & Testing

Now that the Private Endpoint is authorized and DNS is configured, verify that everything is working correctly from the Snowflake side.

### Step 5 — Private Link Allowlist

Retrieve the full list of hostnames and ports that clients need to reach over Private Link. This is useful for validating your DNS configuration and firewall rules.

**Why:** This function returns all endpoints (SNOWFLAKE_DEPLOYMENT, STAGE, OCSP, SNOWSIGHT, etc.) that should resolve to the Private Endpoint IP.

> **Doc:** [SYSTEM$ALLOWLIST_PRIVATELINK](https://docs.snowflake.com/en/sql-reference/functions/system_allowlist_privatelink)

In [ ]:
%%sql -r dataframe_5
SELECT t.VALUE:type::VARCHAR    AS type,
       t.VALUE:host::VARCHAR    AS host,
       t.VALUE:port::INTEGER    AS port
FROM TABLE(
    FLATTEN(
        INPUT => PARSE_JSON(SYSTEM$ALLOWLIST_PRIVATELINK())
    )
) t
ORDER BY type;

### Step 6 — Verify Private Connectivity

Check the current session to confirm you’re connected and see your account’s Private Link URL. If connecting from within the VNet (or via VPN with hosts file), you should use the `privatelink` account URL.

**Why:** Confirms end-to-end connectivity is working. The Private Link account URL has the format: `<account>.<region>.privatelink.snowflakecomputing.com`

> **Tip:** Use [SnowCD (Connectivity Diagnostic Tool)](https://docs.snowflake.com/en/user-guide/snowcd) for comprehensive client-side connectivity testing.

In [ ]:
%%sql -r dataframe_6
SELECT CURRENT_ACCOUNT()  AS account,
       CURRENT_REGION()   AS region,
       CURRENT_SESSION()  AS session_id,
       CURRENT_CLIENT()   AS client_info;

### Step 7 — Check Authorized Endpoints

List all Private Endpoints that have been authorized to connect to this Snowflake account.

**Why:** Useful for auditing which Azure Private Endpoints are approved, especially in multi-endpoint environments.

> **Doc:** [SYSTEM$GET_PRIVATELINK_AUTHORIZED_ENDPOINTS](https://docs.snowflake.com/en/sql-reference/functions/system_get_privatelink_authorized_endpoints)

In [ ]:
%%sql -r dataframe_7
SELECT SYSTEM$GET_PRIVATELINK_AUTHORIZED_ENDPOINTS() AS authorized_endpoints;

---
## Part 6 — Snowflake: Network Policy Lockdown (Recommended)

After confirming Private Link connectivity works, you can optionally block all public access to Snowflake by creating a network policy that only allows traffic from your VNet’s CIDR range.

### Step 8 — Create Network Policy

Create a network policy that restricts access to only your VNet’s IP range. This blocks all public internet access to Snowflake.

**Why:** Azure Private Link alone does not block public access — it only provides an additional private path. A network policy is needed to enforce private-only connectivity.

> **Impact:** This is a **lockout risk**. If Private Link DNS is not configured correctly when you activate this policy, you will be unable to connect. **Test Private Link connectivity thoroughly before applying the policy!**

Replace `<VNET_CIDR>` with your VNet’s address range (e.g., `10.0.0.0/16`).

> **Doc:** [Network policies](https://docs.snowflake.com/en/user-guide/network-policies)

In [ ]:
%%sql -r dataframe_8
CREATE NETWORK POLICY IF NOT EXISTS azure_privatelink_policy
  ALLOWED_IP_LIST = ('<VNET_CIDR>')
  BLOCKED_IP_LIST = ()
  COMMENT = 'Restrict access to Azure Private Link VNet only';

### Step 9 — Activate Network Policy

Apply the network policy at the account level to enforce private-only access.

> **⚠️ LOCKOUT WARNING:** Once activated, **only** connections from the allowed IP range will work. Ensure your current session and all critical access paths are covered by the policy before running this command.

> **Recovery:** If locked out, contact [Snowflake Support](https://docs.snowflake.com/user-guide/contacting-support) to reset the network policy. You can also keep a breakglass IP in the allowed list.

**Why:** This is the final hardening step that ensures all traffic to Snowflake must traverse the Private Endpoint.

> **Impact:** Any client not routing through the VNet (or not in the CIDR range) will receive a connection error.

In [ ]:
%%sql -r dataframe_9
-- UNCOMMENT THE LINE BELOW ONLY AFTER CONFIRMING PRIVATE LINK WORKS
-- ALTER ACCOUNT SET NETWORK_POLICY = azure_privatelink_policy;

-- To verify which policy is active:
SHOW PARAMETERS LIKE 'NETWORK_POLICY' IN ACCOUNT;

---
## Part 7 — Client Configuration

When connecting through Azure Private Link, clients must use the **privatelink** account URL. Below are connection string formats for common clients.

### Connection Strings

| Client | Connection Format |
|---|---|
| **SnowSQL** | `snowsql -a <org>-<account>.privatelink` |
| **JDBC** | `jdbc:snowflake://<acct>.<region>.privatelink.snowflakecomputing.com` |
| **ODBC** | `server=<acct>.<region>.privatelink.snowflakecomputing.com` |
| **Python** | `account='<acct>.<region>.privatelink'` |
| **Snowsight** | `https://app-<org>-<account>.privatelink.snowflakecomputing.com` |
| **.NET** | `account=<acct>.<region>.privatelink;host=<acct>.<region>.privatelink.snowflakecomputing.com` |

### Minimum Driver Versions (Azure Private Link)

| Client | Minimum Version |
|---|---|
| Python Connector | 2.2.8+ |
| JDBC Driver | 3.12.3+ |
| ODBC Driver | 2.21.5+ |
| .NET Driver | 1.2.0+ |
| Go Driver | 1.3.8+ |
| SnowSQL | 1.2.14+ |

> **Tip:** Use the **regionless** URL format (`<org>-<account>.privatelink.snowflakecomputing.com`) for simpler, region-independent connection strings.

---
## Part 8 — Troubleshooting

| Symptom | Likely Cause | Fix |
|---|---|---|
| Private Endpoint stuck in **Pending** state | `SYSTEM$AUTHORIZE_PRIVATELINK` not yet called, or access token expired | Run Step 3 with a fresh access token |
| `Account is not authorized for PrivateLink` | Authorization was never completed or was revoked | Re-run `SYSTEM$AUTHORIZE_PRIVATELINK` |
| Access token error: `Invalid token` | Token expired (max 60 min lifetime) | Regenerate with `az account get-access-token` |
| Token error: `Insufficient permissions` | User lacks Reader role on the subscription | Grant `Reader` role or `Microsoft.Subscription/subscriptions/acceptOwnershipStatus/read` |
| DNS resolution fails from VNet resources | Private DNS Zone not linked to VNet | Run step 4.4 to create the Virtual Network Link |
| DNS resolution fails from end-user machine | Hosts file not updated or VPN not connected | Check hosts file entries and VPN connectivity |
| `nslookup` returns public IP instead of private IP | DNS query not hitting Private DNS Zone | Verify the DNS zone is linked and the record exists; flush DNS cache |
| Snowsight not loading over Private Link | Missing DNS record for `snowsight-privatelink-url` | Add A records for both `app-org-account` and `app.region` URLs |
| OCSP validation errors | Missing DNS record for OCSP URLs | Add A records for `ocsp.*` URLs from the PrivateLink config |
| Connection works but slow | Private Endpoint in a different region than Snowflake | Move PE to the same region or set up VNet peering |
| Locked out after network policy | Network policy CIDR doesn’t include PE IP or DNS wrong | Contact Snowflake Support to reset the network policy |
| `SYSTEM$GET_PRIVATELINK_CONFIG` takes > 1 minute | Known Azure issue | Contact Snowflake Support |
| PLS alias not recognized (red X in Portal) | Alias may be invalid for your region | Contact Snowflake Support for the resource ID value instead |

---
## Part 9 — Cleanup

To tear down the Private Link configuration, run the steps below in reverse order. The SQL commands are commented out to prevent accidental execution.

**Azure cleanup** (run in Azure CLI):

```bash
# Delete the VNet link from the DNS zone
az network private-dns link vnet delete \
  --resource-group <RESOURCE_GROUP_NAME> \
  --zone-name privatelink.snowflakecomputing.com \
  --name snowflake-vnet-link --yes

# Delete DNS A records (repeat for each)
az network private-dns record-set a delete \
  --resource-group <RESOURCE_GROUP_NAME> \
  --zone-name privatelink.snowflakecomputing.com \
  --name <DNS_RECORD_NAME> --yes

# Delete the Private DNS Zone
az network private-dns zone delete \
  --resource-group <RESOURCE_GROUP_NAME> \
  --name privatelink.snowflakecomputing.com --yes

# Delete the Private Endpoint
az network private-endpoint delete \
  --name pep-snowflake-privatelink \
  --resource-group <RESOURCE_GROUP_NAME>

# (Optional) Delete VNet and Resource Group
az network vnet delete --name <VNET_NAME> --resource-group <RESOURCE_GROUP_NAME>
az group delete --name <RESOURCE_GROUP_NAME> --yes
```

### Step 10 — Snowflake Cleanup

Revoke the Private Link authorization and remove the network policy.

> **⚠️ Warning:** Revoking Private Link while the network policy is active will lock out all users. Remove the network policy first!

In [ ]:
%%sql -r dataframe_10
-- ============================================================
-- CLEANUP: Uncomment to tear down Snowflake Private Link config
-- ============================================================

-- Step 1: Remove network policy (if applied)
-- ALTER ACCOUNT UNSET NETWORK_POLICY;

-- Step 2: Drop the network policy
-- DROP NETWORK POLICY IF EXISTS azure_privatelink_policy;

-- Step 3: Revoke Private Link authorization
-- (requires fresh access token from: az account get-access-token --subscription <ID>)
-- SELECT SYSTEM$REVOKE_PRIVATELINK(
--     '<PRIVATE_ENDPOINT_RESOURCE_ID>',
--     '<ACCESS_TOKEN>'
-- );

SELECT 'Cleanup statements are commented out — uncomment to execute' AS status;

---
## References

| Topic | Link |
|---|---|
| Azure Private Link and Snowflake | [docs.snowflake.com](https://docs.snowflake.com/en/user-guide/privatelink-azure) |
| SYSTEM$AUTHORIZE_PRIVATELINK | [docs.snowflake.com](https://docs.snowflake.com/en/sql-reference/functions/system_authorize_privatelink) |
| SYSTEM$GET_PRIVATELINK_CONFIG | [docs.snowflake.com](https://docs.snowflake.com/en/sql-reference/functions/system_get_privatelink_config) |
| SYSTEM$GET_PRIVATELINK | [docs.snowflake.com](https://docs.snowflake.com/en/sql-reference/functions/system_get_privatelink) |
| SYSTEM$REVOKE_PRIVATELINK | [docs.snowflake.com](https://docs.snowflake.com/en/sql-reference/functions/system_revoke_privatelink) |
| SYSTEM$ALLOWLIST_PRIVATELINK | [docs.snowflake.com](https://docs.snowflake.com/en/sql-reference/functions/system_allowlist_privatelink) |
| Azure Private DNS Zones | [docs.microsoft.com](https://docs.microsoft.com/en-us/azure/dns/private-dns-privatednszone) |
| Azure Private Endpoint Overview | [docs.microsoft.com](https://docs.microsoft.com/en-us/azure/private-link/private-endpoint-overview) |
| SnowCD Connectivity Tool | [docs.snowflake.com](https://docs.snowflake.com/en/user-guide/snowcd) |
| Network Policies | [docs.snowflake.com](https://docs.snowflake.com/en/user-guide/network-policies) |
| Pin Private Connectivity Endpoints | [docs.snowflake.com](https://docs.snowflake.com/en/user-guide/pin-private-endpoints) |
| Community: Azure Private DNS Setup | [community.snowflake.com](https://community.snowflake.com/s/article/How-To-Set-up-Private-DNS-zone-for-Azure-Private-Link-with-Snowflake) |